In [2]:
import requests
import io
from warnings import warn
from itertools import zip_longest
from urllib.parse import quote

from bs4 import BeautifulSoup as BS
import numpy as np
import pandas as pd
import polars as pl

In [19]:

class NoItemPageError(Exception):
    pass


class NoBlueprintError(Exception):
    pass


def parse_list_item(li):
    count = int((li := li.next).replace(',', '').replace(" ", ""))
    href = (li := li.next)['href']
    item = li.text

    return {'item': item, 'count': count, 'href': href}


def get_item_materials(name, build_n, stop_items=None):
    stop_items = [] if stop_items is None else stop_items

    page_name = quote(name.title().replace(' ', '_'), safe='/')
    resp = requests.get(f"https://www.starsonata.com/wiki/index.php/{page_name}")

    if 'There is currently no text' in str(resp.content):
        # check for a blueprint page
        page_name += "_Blueprint"
        resp = requests.get(f"https://www.starsonata.com/wiki/index.php/{page_name}")
        if 'There is currently no text' in str(resp.content):
            raise NoItemPageError(f"No page for {name}")
    parsed = BS(resp.content, 'lxml')

    raw = parsed.find("div", "blueprintbox")
    if raw is None:
        raise NoBlueprintError(f"No blueprint information for {name}")

    # get the table information
    bp_info = raw.find_all("td")

    init_raw = bp_info[17]
    per_raw = bp_info[18]

    mats = pl.DataFrame([parse_list_item(i) for i in init_raw.find_all("li")]).join(
        pl.DataFrame([parse_list_item(i) for i in per_raw.find_all("li")]),
        on=['item', 'href'],
        how='full',
        suffix=" periodic",
        coalesce=True
    ).rename({'count': 'count initial'})

    mats = mats.with_columns(name=pl.lit(name))
    # add a total amount column
    mats = mats.with_columns(
        pl.col("count initial") * build_n,
        pl.col("count periodic") * build_n
    ).with_columns(
        **{"count total": pl.sum_horizontal("count initial", "count periodic")}
    )

    # recurse and get sub-components
    sub_mats = []
    final_mats = []
    for row in mats.iter_rows(named=True):
        if row['item'] in stop_items:
            final_mats.append(pl.DataFrame(row))
        elif (row['item'] != 'Credits'):# and ('#' not in row['href']):
            try:
                sub_mats.append(get_item_materials(row['item'], row['count total'], stop_items))
            except (NoBlueprintError, NoItemPageError):
                final_mats.append(pl.DataFrame(row))
        else:
            final_mats.append(pl.DataFrame(row))
    
    sub_mat_res = tuple(
        pl.concat(tuple(final_mats) + i, how='diagonal') for i in zip_longest(*sub_mats, fillvalue=pl.DataFrame())
    )

    return (mats,) + sub_mat_res


In [20]:
sca = get_item_materials("Ruined Haraka", 1, stop_items=['Empyreal Alloy', 'Empyreal Steel'])

In [21]:
sca[0]

item,count initial,href,count periodic,name,count total
str,i64,str,i64,str,i64
"""Credits""",15000000000,"""/wiki/index.php/Credits""",15000000000,"""Ruined Haraka""",30000000000
"""Lunaria Targeting System""",2,"""/wiki/index.php/Lunaria_Target…",null,"""Ruined Haraka""",2
"""Selenic Concealment System""",1,"""/wiki/index.php/Selenic_Concea…",null,"""Ruined Haraka""",1
"""Celestial Flame Infuser""",1,"""/wiki/index.php/Celestial_Flam…",null,"""Ruined Haraka""",1
"""Lunaria Rampage Preserver""",1,"""/wiki/index.php/Lunaria_Rampag…",null,"""Ruined Haraka""",1
…,…,…,…,…,…
"""Selenium""",10,"""/wiki/index.php/Commodities#Se…",null,"""Ruined Haraka""",10
"""Metallic Shard""",5,"""/wiki/index.php/Commodities#Me…",null,"""Ruined Haraka""",5
"""Steel Girders""",null,"""/wiki/index.php/Commodities#St…",1000000,"""Ruined Haraka""",1000000


In [22]:
with pl.Config(tbl_rows=1000, thousands_separator=','):
    display(sca[1].group_by(['item']).agg(pl.col("count initial", "count periodic", "count total").sum()))

item,count initial,count periodic,count total
str,i64,i64,i64
"""Selenic Crystals""",0,1,1
"""Ult. Targeting Augmenter""",0,4,4
"""Credits""","23,000,000,000","23,000,000,000","46,000,000,000"
"""Fusion Cells""",0,"50,000","50,000"
"""Steel Girders""",0,"1,110,000","1,110,000"
"""Ult. Capacity Augmenter""",0,1,1
"""Ult. Aggravation Augmenter""",0,1,1
"""Empyreal Alloy""",0,5,5
"""Sub-Shield Reactors""",0,"2,000","2,000"


In [23]:
with pl.Config(tbl_rows=1000, thousands_separator=','):
    display(sca[2].group_by(['item']).agg(pl.col("count initial", "count periodic", "count total").sum()))

item,count initial,count periodic,count total
str,i64,i64,i64
"""Eridium""",0,10,10
"""Fusion Cells""",0,"50,000","50,000"
"""Selenium""",0,2,2
"""Ult. Targeting Augmenter""",0,4,4
"""Ult. Fading Augmenter""",0,1,1
"""Celestial Shard""",0,5,5
"""Ult. Firing Augmenter""",0,2,2
"""Ult. Capacity Augmenter""",0,1,1
"""Empyreal Alloy""",0,5,5


In [18]:
with pl.Config(tbl_rows=1000, thousands_separator=','):
    display(mats1.group_by("item", "href").agg(pl.col("count initial", "count periodic").sum()))

NameError: name 'mats1' is not defined

In [159]:
a = [(5, 10), (15,), (25, 30)]

for i in zip_longest(*a, fillvalue=None):
    print(i)

(5, 15, 25)
(10, None, 30)


In [5]:
get_item_materials("Empyreal_Launch_Assembly_Blueprint", 1)

NameError: name 'NoBlueprintError' is not defined

In [31]:
heat_damp = get_item_materials("Laconia Reinforced Heat Dampener+ Blueprint", 15)
surg_damp = get_item_materials("Laconia Reinforced Surgical Dampener+ Blueprint", 15)
# rad_damp = get_item_materials("Laconia Reinforced Radiation Dampener+ Blueprint", 15)
energy_damp = get_item_materials("Laconia Reinforced Energy Dampener+ Blueprint", 15)
mining_damp = get_item_materials("Laconia Reinforced Mining Dampener+ Blueprint", 15)
phys_damp = get_item_materials("Laconia Reinforced Physical Dampener+ Blueprint", 15)

In [33]:
with pl.Config(tbl_rows=1000, thousands_separator=','):
    display(heat_damp[1].group_by(['item']).agg(pl.col("count initial", "count periodic", "count total").sum()))

item,count initial,count periodic,count total
str,i64,i64,i64
"""Exc. Shield Booster""",15,0,15
"""Credits""","2,250,000,000","2,250,000,000","4,500,000,000"
"""Heat Dampener+""",30,0,30
"""Titanium Sheet""",300,0,300
"""Good Shield Booster""",15,0,15
"""Metals""","1,650,000","1,650,000","3,300,000"
"""Laconia Sheet""",150,0,150


In [34]:
with pl.Config(tbl_rows=1000, thousands_separator=','):
    display(heat_damp[2].group_by(['item']).agg(pl.col("count initial", "count periodic", "count total").sum()))

item,count initial,count periodic,count total
str,i64,i64,i64
"""Credits""","5,250,000,000","3,750,000,000","9,000,000,000"
"""Heat Dampener""",0,60,60
"""Titanium Sheet""",300,0,300
"""Metals""","4,650,000","1,650,000","6,300,000"
"""Hell Fire Remains""",30,0,30
"""Good Shield Booster""",15,0,15
"""Laconia Sheet""",150,0,150
"""Exc. Shield Booster""",15,0,15


In [32]:
with pl.Config(tbl_rows=1000, thousands_separator=','):
    display(heat_damp[-1].group_by(['item']).agg(pl.col("count initial", "count periodic", "count total").sum()))

item,count initial,count periodic,count total
str,i64,i64,i64
"""Credits""","5,850,000,000","4,050,000,000","9,900,000,000"
"""Hell Fire Remains""",90,0,90
"""Good Shield Booster""",15,0,15
"""Metals""","5,250,000","1,650,000","6,900,000"
"""Titanium Sheet""",300,0,300
"""Laconia Sheet""",150,0,150
"""Exc. Shield Booster""",15,0,15


In [5]:
blossom = get_item_materials("Burning Blossom Blueprint", 1)

In [8]:
with pl.Config(tbl_rows=1000, thousands_separator=','):
    display(blossom[-1].group_by(['item']).agg(pl.col("count initial", "count periodic", "count total").sum()))

item,count initial,count periodic,count total
str,i64,i64,i64
"""Burning Petals""",20,0,20
"""Venusian Magcannon""",10,0,10
"""Credits""","9,700,000,000","9,700,000,000","19,400,000,000"
"""Metals""","8,500,000",0,"8,500,000"
"""Death Warmed Over""",0,4,4
"""Solarian MagCannon""",1,0,1
"""Reach of Prometheus""",0,4,4
"""Nutter Sputters""",0,2,2
"""Energon""",0,50,50
